In [ ]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.utils import *
from module.prompt import *
from module.custom_model import *
from module.tools import *
from module.db_agent import *
from module.data_analysis_agent import *
from module.conversaction_agent import *

from typing_extensions import TypedDict

from typing import Annotated, List, Literal, Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    RemoveMessage,
)
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph_supervisor import create_supervisor


class State(TypedDict):
    question: Annotated["str", "human request to llm"]
    summary: Annotated[str, "previous messages summarization "]
    messages: Annotated[list, add_messages]  # 화면 출력용
    answer: Annotated["str", "llm generate answer"]


def get_prompt_summary():
    template = """ 
        #System :
        As a summary-only node in LangGraph, you are responsible for summarizing key information by receiving records from previous conversations.  
        Be sure to follow the following rules:

        # Rules:
        1. If the user's requirements are **Analyzing, Comparison, Evaluation, and Modification requests for answers generated by previous LLM:
        - Never summarize the answers generated by the previous LLM and keep the original ****.
        - Combine content by briefly summarizing other surrounding contexts (e.g., discussion process, question background, etc.)

        2. For common conversations (information queries, descriptions, requests, etc.) that do not meet the above conditions:
        - Only important contents are summarized in **sentence form.**.
        - Remove unnecessary detailed descriptions, duplicate sentences, and text repeated in context.
        - It is described focusing on facts, and opinions or expressions of emotions are excluded.

        3. Finally, the summary is written in Korean.

        # Enter :
        - User Request : {question}
        - LLM's previous answer : {answer}
        - All previous conversations : {messages}
        - Summary of the conversation : {summary}
    """
    return ChatPromptTemplate.from_template(template)


def summary_node(state: State):
    """
    이전 모든 대화 내용중 핵심을 포함하여 요약하기위한 노드
    messages 가 6개 이상인 경우 summarization 수행
    """
    question = state.get("question", "")
    messages = state.get("messages", "")
    answer = state.get("answer", "")
    summary = state.get("summary", "")
    chain = get_prompt_summary() | get_gemini()
    response = chain.invoke(
        {
            "question": question,
            "messages": messages,
            "answer": answer,
            "summary": summary,
        }
    )
    if len(messages) > 2:
        # 오래된 메시지 삭제
        delete_messages = [RemoveMessage(id=m.id) for m in state["messages"]]
        # 요약 정보 반환
        return {
            "summary": response.content,
            "messages": delete_messages,
        }
    else:
        return {
            "summary": response.content,
        }


def supervisor_node(state: State):
    """
    두개의 에이전트를 자율적으로 선택하여 사용하는 관리자에이전트 구조
    """
    summary = state.get("summary", "")
    question = state["question"]
    content = f"Previous Conversations: {summary} \n Human Request : {question}"
    db = get_db_agent()
    conversation = get_conversation_agent()
    data_analysis = get_data_analysis_agent()
    supervisor = create_supervisor(
        model=get_gpt(),
        agents=[db, conversation, data_analysis],
        prompt=(
            "You are a supervisor managing three agents:\n"
            "- db_agent: assign database select,insert,delete,update tasks\n"
            "- conversation_agent: assign normal conversaction and no use agent\n"
            "- data_analysis_agent: Agents that generate and run Python code for data analysis\n"
            "# Important : you must final answer is Korean\n"
        ),
        add_handoff_back_messages=True,
        output_mode="full_history",
    ).compile()
    inputs = {"messages": [{"role": "user", "content": content}]}
    config = get_runnable_config(recursion_limit=15, thread_id=get_random_uuid())
    fianl_mode=''
    final_data=''
    for mode, data in supervisor.stream(
        inputs, config, stream_mode=["values", "messages"]
    ):
        fianl_mode = mode
        final_data = data
        
    return {'messages':final_data['messages'],'answer':final_data['messages'][-1]}



def get_graph():
    state_graph = StateGraph(State)
    state_graph.add_node("summary_node", summary_node)
    state_graph.add_node("supervisor_node", supervisor_node)

    state_graph.add_edge(START, "summary_node")
    state_graph.add_edge("summary_node", "supervisor_node")
    state_graph.add_edge("supervisor_node", END)

    cp = get_check_pointer()
    return state_graph.compile(checkpointer=cp)


def get_config():
    return get_runnable_config(recursion_limit=10, thread_id=get_random_uuid())

In [ ]:
graph = get_graph()
config = get_config()
user_inputs = "내이름은 마이클 조던이야"
inputs = inputs = {"question": user_inputs}
for chunk_msg, metadata in graph.stream(
    inputs, config, stream_mode="messages", subgraphs=True
):
    if chunk_msg != ():
        _metadata = metadata[0]
        print(_metadata.content,end='')
print(f'snapshot : \n {graph.get_state(config).values}\n')


안녕하세요, 마이클 조던! 이름이 정말 멋지네요. 오늘 기분은 어떠세요?안녕하세요, 마이클 조던님! 무엇을 도와드릴까요?snapshot : 
 {'question': '내이름은 마이클 조던이야', 'summary': "- User Request : 내이름은 마이클 조던이야\n        - LLM's previous answer : \n        - All previous conversations : []\n        - Summary of the conversation : 사용자의 이름은 마이클 조던입니다.", 'messages': [HumanMessage(content="Previous Conversations: - User Request : 내이름은 마이클 조던이야\n        - LLM's previous answer : \n        - All previous conversations : []\n        - Summary of the conversation : 사용자의 이름은 마이클 조던입니다. \n Human Request : 내이름은 마이클 조던이야", additional_kwargs={}, response_metadata={}, id='57b018c4-84eb-4fa2-824f-b2ccd359e8d6'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_LLf2hA1AHOZoio97zxcHREit', 'function': {'arguments': '{}', 'name': 'transfer_to_conversation_agent'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'service_tier': 

In [17]:
user_inputs = "내 직업은 농구 선수고 "
inputs = inputs = {"question": user_inputs}
for chunk_msg, metadata in graph.stream(
    inputs, config, stream_mode="messages", subgraphs=True
):
    if chunk_msg != ():
        _metadata = metadata[0]
        print(_metadata.content,end='')
print(f'snapshot : \n {graph.get_state(config).values}\n')

네, 마이클 조던 선수! 농구 선수로서 어떤 점이 가장 즐겁고, 또 어떤 점이 가장 힘든가요? 이야기를 들어보고 싶어요.마이클 조던 선수, 농구 선수로서 어떤 점이 가장 즐겁고, 또 어떤 점이 가장 힘든지 이야기해 주실 수 있나요?snapshot : 
 {'question': '내 직업은 농구 선수고 ', 'summary': '사용자의 이름은 마이클 조던이고, 직업은 농구 선수입니다.', 'messages': [HumanMessage(content='Previous Conversations: 사용자의 이름은 마이클 조던이고, 직업은 농구 선수입니다. \n Human Request : 내 직업은 농구 선수고 ', additional_kwargs={}, response_metadata={}, id='144a6278-ca3e-4b4d-b67c-0356f93d1e8f'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_nedEPRwUb2A90GmvzXS6QzZJ', 'function': {'arguments': '{}', 'name': 'transfer_to_conversation_agent'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'service_tier': 'default'}, name='supervisor', id='run--66037a11-6546-49b6-a818-c0e36e7e9b0e', tool_calls=[{'name': 'transfer_to_conversation_agent', 'args': {}, 'id': 'call_nedEPRwUb2A90GmvzXS6QzZJ', 'type': 'tool_call'}]), ToolM

In [18]:
user_inputs = "요즘에는 화가로 일하고 있어"
inputs = inputs = {"question": user_inputs}
for chunk_msg, metadata in graph.stream(
    inputs, config, stream_mode="messages", subgraphs=True
):
    if chunk_msg != ():
        _metadata = metadata[0]
        print(_metadata.content,end='')


print(f'snapshot : \n {graph.get_state(config).values}\n')

아, 마이클 조던 씨가 이제 화가로 활동하고 계시다니 정말 멋지네요! 농구 선수에서 화가로 전향한 이야기도 궁금한데, 어떤 스타일의 그림을 주로 그리시나요?마이클 조던 씨는 농구 선수에서 현재는 화가로 활동하고 계십니다.snapshot : 
 {'question': '요즘에는 화가로 일하고 있어', 'summary': '사용자의 이름은 마이클 조던입니다. 그는 농구 선수였지만, 현재는 화가로 일하고 있습니다.', 'messages': [HumanMessage(content='Previous Conversations: 사용자의 이름은 마이클 조던입니다. 그는 농구 선수였지만, 현재는 화가로 일하고 있습니다. \n Human Request : 요즘에는 화가로 일하고 있어', additional_kwargs={}, response_metadata={}, id='69c83ed9-247c-410d-abf6-3efa96166df3'), AIMessage(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_KGM0oF9vQoqBlDtYw1mGbwbS', 'function': {'arguments': '{}', 'name': 'transfer_to_conversation_agent'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_c064fdde7c', 'service_tier': 'default'}, name='supervisor', id='run--4c6ccbeb-675d-4e43-8c49-b5cc0737217b', tool_calls=[{'name': 'transfer_to_conversation_agent', 'args': {}, 'id': 'call_KGM0oF9vQoqBlDtYw1mGbw